# Phase 2 Part B: Structured Prompt Engineering & Generative AI

**Project:** CCO Flight Booking & Passenger Advisory Dashboard (SWE485, Group 5)

This notebook turns the Phase 1 booking-completion predictions into short executive briefings with a Generative AI model. Each briefing explains, in plain business language, how likely a booking is to be completed, which booking details matter, and what the commercial team could do.

| Section | Owner |
| --- | --- |
| 1. Generative AI Model Selection & Setup | Member C |
| 2. Preparing Model Outputs for the Prompts | Member C |
| 3. Template 1 Design Documentation | Member C |
| 4. Template 1 Testing | Member C |
| 5. Template 2 Design Documentation | Member D |
| 6. Testing Framework & Output Comparison | Member D |
| 7. Analysis: Qualitative & Quantitative Results | Member D |
| 8. Best Prompt Selection & Justification | Member D |
| 9. Integration Plan for the Final System | Members D and E |
| 10. Ethical Considerations & Limitations | Member D |

## 1. Generative AI Model Selection & Setup

### 1.1 Model choice

We use **Google Gemini 3.6 Flash** (`gemini-3.6-flash`) through the official `google-genai` Python SDK.

We first planned Gemini 2.5 Flash, but Google no longer offers it to new API keys. When we tested the newest versions (3.7 and 3.8 Flash), they returned "high demand" errors. 3.6 Flash answered reliably, so we use it. The model name is a single setting in `api_config_template.py` (or the `GEMINI_MODEL` environment variable), so it can be changed without touching the templates.

| Criterion | Why Gemini 3.6 Flash fits this project |
| --- | --- |
| Cost and access | The Google AI Studio free tier is enough for our test cases and the dashboard demo, so no team member needs to pay. |
| Instruction following | Our templates ask for a fixed structure and strict grounding rules. Current Gemini models follow system instructions and output formats reliably. |
| Speed | Flash models answer in a few seconds, which suits an interactive dashboard where an executive changes a scenario and waits for the briefing. |
| No local hardware | Open models such as LLaMA would need a GPU or a slow CPU setup in Colab. A hosted API keeps the notebook runnable for every member. |
| System instructions | The API separates the system instruction (role and rules) from the user prompt (booking data), which matches how we document each template. |

**Alternatives considered.** GPT-4-class models give similar quality but have no free API tier. A local LLaMA model avoids sending data to an external service, but our data is a public, anonymous Kaggle dataset, so privacy is not a blocker, and the hardware cost would slow the team down.

### 1.2 Generation settings

Both templates use the same settings so that the comparison in Section 6 is fair:

- `temperature = 0.2`: low randomness keeps the briefings factual and makes reruns similar.
- `max_output_tokens = 2048`: far more than a 200-word briefing needs, so answers are not cut off.
- `thinking_level = "minimal"`: Gemini 3 models reason internally before answering. The task is a short rewrite of given facts, so minimal thinking keeps answers fast without lowering quality. In our tests, the "low" level used about 540 hidden thinking tokens for a 150-word answer.

### 1.3 Safe API key handling

The key is **never written in code or committed to GitHub**:

- `Generative_AI/api_config_template.py` contains only the code that *looks for* the key: first an environment variable, then Colab Secrets, then a local `.env` file.
- `.env.example` (repository root) shows the expected variable name with an empty value.
- `.gitignore` excludes `.env`, `.env.*` (except the example), `*.key`, `api_keys.*`, `Generative_AI/api_config.py` and `Generative_AI/secrets/`.
- The notebook never prints the key, and outputs saved to `example_outputs/` contain only prompts and responses.

**To run this notebook:**

- **In Colab:** click the key icon in the left sidebar, add a secret named `GEMINI_API_KEY`, and enable notebook access.
- **Locally:** copy `.env.example` to `.env` and paste your key after `GEMINI_API_KEY=`.

A free key can be created at [Google AI Studio](https://aistudio.google.com/apikey). Without a key, the notebook still builds and shows every prompt; only the live model calls are skipped.

In [1]:
# Install the SDKs when they are missing (Colab already has pandas).
import importlib.util, subprocess, sys
for pkg, module in [("google-genai", "google.genai"), ("python-dotenv", "dotenv")]:
    if importlib.util.find_spec(module.split(".")[0]) is None or importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

In [2]:
import json, os, re, time, urllib.request
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

# The notebook works from the repository folder (local Jupyter) or from GitHub (Colab).
REPO_RAW = "https://raw.githubusercontent.com/Ghaida-Alyousef/SW485-Project-Group5/main"
HERE = Path.cwd()
REPO_ROOT = HERE.parent if (HERE.parent / "Dataset").exists() else HERE
LOCAL = (REPO_ROOT / "Dataset" / "customer_booking.csv").exists()

def repo_path(relative):
    # Local file path when the repository is cloned, otherwise the GitHub raw URL.
    return str(REPO_ROOT / relative) if LOCAL else f"{REPO_RAW}/{relative}"

# Make api_config_template.py importable. In Colab it is downloaded from the repository.
GENAI_DIR = REPO_ROOT / "Generative_AI" if LOCAL else HERE
if not (GENAI_DIR / "api_config_template.py").exists():
    urllib.request.urlretrieve(f"{REPO_RAW}/Generative_AI/api_config_template.py", GENAI_DIR / "api_config_template.py")
sys.path.insert(0, str(GENAI_DIR))

import api_config_template as cfg

PROMPTS_DIR = GENAI_DIR / "prompts"
OUTPUTS_DIR = GENAI_DIR / "example_outputs"
PROMPTS_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)
for name in ["T1_structured_executive_briefing.json"]:
    if not (PROMPTS_DIR / name).exists():
        urllib.request.urlretrieve(f"{REPO_RAW}/Generative_AI/prompts/{name}", PROMPTS_DIR / name)

API_READY = cfg.has_api_key()
print("Repository files found locally:", LOCAL)
print("Model:", cfg.MODEL_NAME)
print("API key available:", API_READY)  # prints True/False only, never the key

Repository files found locally: True
Model: gemini-3.6-flash
API key available: True


In [3]:
def generate(template, user_prompt, retries=3):
    # Send one filled template to Gemini and return (text, seconds taken).
    from google.genai import types

    settings = dict(cfg.GENERATION_SETTINGS)
    settings["thinking_config"] = types.ThinkingConfig(thinking_level=cfg.THINKING_LEVEL)
    config = types.GenerateContentConfig(system_instruction=template["system_instruction"], **settings)

    client = cfg.get_client()
    for attempt in range(1, retries + 1):
        try:
            start = time.perf_counter()
            response = client.models.generate_content(model=cfg.MODEL_NAME, contents=user_prompt, config=config)
            return response.text.strip(), round(time.perf_counter() - start, 2)
        except Exception as error:
            # The free tier allows a limited number of requests per minute, so wait and retry.
            if attempt == retries:
                raise
            print(f"Attempt {attempt} failed ({type(error).__name__}); retrying in 30 seconds...")
            time.sleep(30)

## 2. Preparing Model Outputs for the Prompts

The prompts are filled from the saved Phase 1 outputs, so the Generative AI layer explains the **same predictions** the dashboard shows:

| Input | Source file | Used for |
| --- | --- | --- |
| Original booking values | `Dataset/customer_booking.csv` | Readable booking details (country names, routes, days) instead of scaled numbers |
| Completion probability | `Supervised_Learning/evaluation_results/model1_predictions.csv` | Random Forest probability for each held-out test booking (`row_id` = row in the raw file) |
| Decision threshold | `model1_run_info.json` | The F1-optimal threshold of 0.19 chosen in Phase 1 |
| Model drivers | `model1_feature_importance.csv` | Permutation importance of the strongest features |
| Train/test split | `Dataset/preprocessed_data.csv` (`data_split`) | Historical completion rates are computed from **training rows only**, so test labels never enter a prompt |
| Passenger segment | `Unsupervised_Learning/clustered_data.csv` and `clustering_results/segment_profiles.csv` (Part A) | The booking's K-Means segment and that segment's profile |

In [4]:
raw = pd.read_csv(repo_path("Dataset/customer_booking.csv"), encoding="latin-1")
split = pd.read_csv(repo_path("Dataset/preprocessed_data.csv"), usecols=["data_split"])
predictions = pd.read_csv(repo_path("Supervised_Learning/evaluation_results/model1_predictions.csv"))
importance = pd.read_csv(repo_path("Supervised_Learning/evaluation_results/model1_feature_importance.csv"))
with urllib.request.urlopen(repo_path("Supervised_Learning/evaluation_results/model1_run_info.json")) if not LOCAL else open(repo_path("Supervised_Learning/evaluation_results/model1_run_info.json")) as f:
    run_info = json.load(f)

THRESHOLD = run_info["tuned_threshold"]
raw["data_split"] = split["data_split"].values
train = raw[raw["data_split"] == "train"]
BASELINE_RATE = train["booking_complete"].mean()

# Sanity check: prediction rows point at the right raw bookings.
assert (raw.loc[predictions["row_id"], "booking_complete"].values == predictions["y_true"].values).all()
print(f"Test predictions: {len(predictions):,} | threshold: {THRESHOLD} | training completion rate: {BASELINE_RATE:.1%}")

Test predictions: 10,011 | threshold: 0.19 | training completion rate: 14.9%


### 2.1 Passenger segments from the unsupervised part

The unsupervised part chose **K-Means with 7 segments**. Each booking's segment is in `Unsupervised_Learning/clustered_data.csv` (column `cluster`, matched by `row_id`), and each segment's name and statistics are in `Unsupervised_Learning/clustering_results/segment_profiles.csv`.

The segment text in the prompt is built **only from the numbers in that file** (size, completion rate and typical booking pattern). We deliberately leave out the "suggested action" lines from `cluster_profiles.txt`, because some of them mention prices, and Template 1 forbids price advice: the dataset has no price information. The model writes its own recommendation from the facts instead.

If the files are missing, the segment placeholder says that segment information is not available, and the template is instructed to say so rather than guess.

In [5]:
def describe_segment(s):
    # One factual sentence block per segment, built from segment_profiles.csv.
    return (f"\"{s['name']}\" (segment {s['segment']} of 7): {s['bookings']:,} bookings ({s['share_pct']:.1f}% of all). "
            f"Completion rate {s['completion_rate']:.1%}. Typical booking: {s['median_purchase_lead']:.0f} days ahead, "
            f"{s['median_length_of_stay']:.0f}-day stay, {s['mean_passengers']:.1f} passengers. "
            f"Requests extra baggage {s['extra_baggage_rate']:.0%}, preferred seat {s['preferred_seat_rate']:.0%}, "
            f"meals {s['meal_rate']:.0%} of the time.")

try:
    clustered = pd.read_csv(repo_path("Unsupervised_Learning/clustered_data.csv"), usecols=["row_id", "cluster"])
    CLUSTER_OF_ROW = clustered.set_index("row_id")["cluster"]
    profiles = pd.read_csv(repo_path("Unsupervised_Learning/clustering_results/segment_profiles.csv"))
    SEGMENT_PROFILES = {int(s["segment"]): describe_segment(s) for _, s in profiles.iterrows()}
    print(f"Segments loaded for {len(CLUSTER_OF_ROW):,} bookings; {len(SEGMENT_PROFILES)} segment profiles.")
except Exception as error:
    CLUSTER_OF_ROW, SEGMENT_PROFILES = None, {}
    print(f"Clustering results not available ({type(error).__name__}); segment information will be marked as missing.")

def segment_text(row_id):
    if CLUSTER_OF_ROW is None or row_id not in CLUSTER_OF_ROW.index:
        return "Segment information is not available yet (clustering results pending)."
    return SEGMENT_PROFILES.get(int(CLUSTER_OF_ROW[row_id]), "No profile is available for this segment.")

for seg, text in SEGMENT_PROFILES.items():
    print(f"- {text}")

Segments loaded for 50,000 bookings; 7 segment profiles.
- "Early planners" (segment 0 of 7): 4,772 bookings (9.5% of all). Completion rate 14.2%. Typical booking: 269 days ahead, 6-day stay, 1.7 passengers. Requests extra baggage 54%, preferred seat 21%, meals 27% of the time.
- "Top market (Australia)" (segment 1 of 7): 13,272 bookings (26.5% of all). Completion rate 5.4%. Typical booking: 48 days ahead, 22-day stay, 1.3 passengers. Requests extra baggage 73%, preferred seat 34%, meals 55% of the time.
- "Short trips, few extras" (segment 2 of 7): 16,075 bookings (32.1% of all). Completion rate 16.5%. Typical booking: 36 days ahead, 6-day stay, 1.3 passengers. Requests extra baggage 45%, preferred seat 6%, meals 10% of the time.
- "Group travellers" (segment 3 of 7): 2,792 bookings (5.6% of all). Completion rate 16.7%. Typical booking: 98 days ahead, 6-day stay, 4.7 passengers. Requests extra baggage 84%, preferred seat 32%, meals 48% of the time.
- "Top route (AKLKUL)" (segment 4 of

### 2.2 Turning a booking into prompt placeholders

`build_case` converts one test booking into the values of the template placeholders. Design choices:

- **Readable values.** The prompt uses the original values (for example "Malaysia", "Wednesday at 14:00") rather than encoded or scaled numbers, so the model does not have to interpret preprocessing.
- **Likelihood tiers.** The probability is also given as a tier, because executives act on categories more easily than on raw probabilities:
  - **Low**: below the 0.19 decision threshold. The model expects the booking to be abandoned.
  - **Moderate**: from 0.19 up to 0.40. The booking is flagged as likely to complete, but the outcome is uncertain.
  - **High**: 0.40 or more, which is well above the 15% average.
- **Historical context with counts.** The completion rate for the booking's origin and route is reported with the number of training bookings behind it, so the model (and the reader) can tell a stable rate from one based on a handful of bookings.
- **Global drivers.** Phase 1 computed permutation importance for the whole model, not per-booking explanations. The prompt therefore lists the strongest drivers *overall* with this booking's values, and the template tells the model that these are associations.

In [6]:
DAY_NAMES = {"Mon": "Monday", "Tue": "Tuesday", "Wed": "Wednesday", "Thu": "Thursday",
             "Fri": "Friday", "Sat": "Saturday", "Sun": "Sunday"}

# Readable names for the encoded feature columns used in Phase 1.
DRIVER_LABELS = {
    "booking_origin": ("Booking origin (country the booking was made from)", lambda b: b["booking_origin"]),
    "route": ("Route", lambda b: b["route"]),
    "flight_duration": ("Flight duration", lambda b: f"{b['flight_duration']} hours"),
    "length_of_stay": ("Length of stay", lambda b: f"{b['length_of_stay']} days"),
    "total_extra_services": ("Number of extra services requested", lambda b: str(b["wants_extra_baggage"] + b["wants_preferred_seat"] + b["wants_in_flight_meals"])),
    "purchase_lead": ("Days between booking and departure", lambda b: f"{b['purchase_lead']} days"),
    "sales_channel": ("Sales channel", lambda b: b["sales_channel"]),
    "wants_preferred_seat": ("Requested a preferred seat", lambda b: "yes" if b["wants_preferred_seat"] else "no"),
    "wants_extra_baggage": ("Requested extra baggage", lambda b: "yes" if b["wants_extra_baggage"] else "no"),
    "wants_in_flight_meals": ("Requested in-flight meals", lambda b: "yes" if b["wants_in_flight_meals"] else "no"),
    "flight_hour": ("Departure hour", lambda b: f"{b['flight_hour']:02d}:00"),
    "num_passengers": ("Number of passengers", lambda b: str(b["num_passengers"])),
}

def base_feature(name):
    # Map one-hot columns such as sales_channel_Mobile back to their original feature.
    for prefix in ("sales_channel", "trip_type", "flight_day"):
        if name.startswith(prefix):
            return prefix
    return name

# Top five original features by permutation importance (one-hot columns merged).
driver_rank = (importance.assign(feature=importance["feature"].map(base_feature))
               .groupby("feature")["permutation_importance_mean"].sum()
               .sort_values(ascending=False))
TOP_DRIVERS = [f for f in driver_rank.index if f in DRIVER_LABELS][:5]

def group_rate(column, value):
    group = train[train[column] == value]["booking_complete"]
    return group.mean(), len(group)

def likelihood_tier(p):
    if p < THRESHOLD:
        return "Low (below the decision threshold; the booking is expected to be abandoned)"
    if p < 0.40:
        return "Moderate (above the threshold, but the outcome is uncertain)"
    return "High (well above the average completion rate)"

def build_case(row_id):
    b = raw.loc[row_id]
    p = float(predictions.loc[predictions["row_id"] == row_id, "y_proba"].iloc[0])
    services = [name for col, name in [("wants_extra_baggage", "extra baggage"),
                                         ("wants_preferred_seat", "a preferred seat"),
                                         ("wants_in_flight_meals", "in-flight meals")] if b[col]]
    profile = "\n".join([
        f"- Passengers: {b['num_passengers']}",
        f"- Sales channel: {b['sales_channel']}",
        f"- Trip type: {b['trip_type']}",
        f"- Booked {b['purchase_lead']} days before departure",
        f"- Length of stay: {b['length_of_stay']} days",
        f"- Flight: {DAY_NAMES[b['flight_day']]} at {b['flight_hour']:02d}:00 on route {b['route']} ({b['flight_duration']} hours)",
        f"- Booking origin: {b['booking_origin']}",
        f"- Extra services requested: {', '.join(services) if services else 'none'}",
    ])
    origin_rate, origin_n = group_rate("booking_origin", b["booking_origin"])
    route_rate, route_n = group_rate("route", b["route"])
    drivers = "\n".join(f"{i}. {DRIVER_LABELS[f][0]}: this booking = {DRIVER_LABELS[f][1](b)}"
                        for i, f in enumerate(TOP_DRIVERS, 1))
    return {
        "row_id": int(row_id),
        "actual_outcome": int(b["booking_complete"]),  # kept for evaluation only, never put in the prompt
        "booking_profile": profile,
        "completion_probability": f"{p:.1%}",
        "decision_threshold": f"{THRESHOLD:.0%}",
        "predicted_outcome": "Likely to be completed" if p >= THRESHOLD else "Likely to be abandoned",
        "likelihood_tier": likelihood_tier(p),
        "baseline_rate": f"{BASELINE_RATE:.1%}",
        "origin_context": f"- Bookings from {b['booking_origin']}: {origin_rate:.1%} completed ({origin_n:,} training bookings)",
        "route_context": f"- Bookings on route {b['route']}: {route_rate:.1%} completed ({route_n:,} training bookings)",
        "top_model_drivers": drivers,
        "segment_profile": segment_text(row_id),
    }

### 2.3 Test case selection

The handbook asks for at least three test cases per template. We pick three held-out **test** bookings that cover the three likelihood tiers, so each template is checked on a clear "abandon" case, a borderline case, and a clear "complete" case:

| Case | Rule |
| --- | --- |
| 1. Low | Probability closest to 5% among bookings that were actually abandoned |
| 2. Borderline | Probability closest to 25%, just above the 0.19 threshold |
| 3. High | Probability closest to 55% among bookings that were actually completed |

Each case must come from a different booking origin, so the briefings cannot all repeat the same context. The selection is deterministic, and the chosen `row_id`s are saved to `example_outputs/test_cases.json`. **Template 2 must use the same three cases** so the comparison is fair.

In [7]:
def pick(target, pool, used_origins):
    pool = pool[~raw.loc[pool["row_id"], "booking_origin"].isin(used_origins).values]
    return int(pool.iloc[(pool["y_proba"] - target).abs().argsort().iloc[0]]["row_id"])

used = set()
TEST_CASES = {}
for name, target, pool in [
    ("Case 1 - Low", 0.05, predictions[predictions["y_true"] == 0]),
    ("Case 2 - Borderline", 0.25, predictions),
    ("Case 3 - High", 0.55, predictions[predictions["y_true"] == 1]),
]:
    row_id = pick(target, pool, used)
    used.add(raw.loc[row_id, "booking_origin"])
    TEST_CASES[name] = build_case(row_id)

with open(OUTPUTS_DIR / "test_cases.json", "w", encoding="utf-8") as f:
    json.dump(TEST_CASES, f, indent=2, ensure_ascii=False)

pd.DataFrame([{"case": k, "row_id": v["row_id"], "origin": raw.loc[v["row_id"], "booking_origin"],
               "route": raw.loc[v["row_id"], "route"], "probability": v["completion_probability"],
               "model decision": v["predicted_outcome"], "actual": "completed" if v["actual_outcome"] else "abandoned"}
              for k, v in TEST_CASES.items()])

,case,row_id,origin,route,probability,model decision,actual
0,Case 1 - Low,9669,Australia,DMKOOL,5.0%,Likely to be abandoned,abandoned
1,Case 2 - Borderline,11278,Vietnam,HANSYD,25.0%,Likely to be completed,abandoned
2,Case 3 - High,35092,Malaysia,AKLKUL,55.0%,Likely to be completed,completed


## 3. Template 1 Design Documentation

| Field | Description |
| --- | --- |
| **Template name** | Structured Executive Briefing |
| **Template ID** | `T1` (file: `prompts/T1_structured_executive_briefing.json`) |
| **Intended use case** | The default briefing on the dashboard: when the CCO selects a booking or changes a scenario, T1 explains the completion outlook in under a minute of reading. |
| **Technique** | Zero-shot role prompting with a fixed output schema and explicit grounding rules |

### 3.1 Design rationale

1. **Role in the system instruction.** "Commercial analytics advisor writing for the CCO" sets the audience and tone once, separate from the data. This keeps the user prompt purely factual, and the same instruction applies to every booking.
2. **Fixed five-heading schema.** Executives scan rather than read. Every briefing has the same order (Headline, Outlook, Key factors, Recommended action, Confidence and caveats), so the CCO always knows where to look, and the dashboard can show the headline on its own. A fixed format also makes outputs comparable across bookings and easy to check automatically.
3. **Grounding rules against hallucination.** The model may use only the supplied facts, must call drivers "associations", and may not mention prices, because the dataset has none. These rules target the most likely failure modes we expected: invented numbers, causal claims, and price advice the data cannot support.
4. **Context instead of raw features.** Giving the origin and route completion rates (with counts) lets the model compare this booking with history without doing any calculation itself. Language models are weak at arithmetic but good at explaining numbers they are given.
5. **Action with an owner.** Asking for a responsible team (for example digital sales) makes recommendations concrete enough to assign, which is what a decision-support briefing is for.
6. **Length limit.** The prompt asks for under 180 words, leaving a margin below the 200 words that fit a dashboard panel. Language models count words loosely, so the margin is needed (see Section 4.1).

**Why zero-shot rather than few-shot?** A worked example in the prompt tends to make the model copy the example's wording and recommendations for every booking. With a clear schema, zero-shot keeps briefings specific to each booking and keeps the prompt short and cheap. Template 2 can test a different approach.

### 3.2 Prompt structure

The template has two parts: a **system instruction** (role and rules, the same for every booking) and a **user template** with placeholders filled by `build_case`.

In [8]:
with open(PROMPTS_DIR / "T1_structured_executive_briefing.json", encoding="utf-8") as f:
    T1 = json.load(f)

display(Markdown("**System instruction**\n\n```text\n" + T1["system_instruction"] + "\n```"))
display(Markdown("**User template**\n\n```text\n" + T1["user_template"] + "\n```"))

# Every placeholder in the template must be produced by build_case.
missing = set(re.findall(r"{(\w+)}", T1["user_template"])) - set(next(iter(TEST_CASES.values())))
assert not missing, f"Placeholders without a value: {missing}"
print("Placeholders:", ", ".join(T1["placeholders"]))

**System instruction**

```text
You are a commercial analytics advisor writing for the Chief Commercial Officer (CCO) of an airline. You explain the output of a machine learning model that estimates whether a flight booking will be completed.

Rules:
- Use only the facts given in the input. Do not invent numbers, prices, customer names, competitors, or causes.
- Model drivers are associations learned from historical data, not proven causes. Say "associated with", never "caused by".
- The data contains no ticket prices. Do not recommend price changes or discounts based on price.
- If a section of the input is marked as not available, say so in a short phrase and continue. Never describe information as missing when it is given in the input.
- Describe the model's accuracy honestly: it is useful for prioritising bookings, but it is not certain for any single booking.
- Write in plain business English for a non-technical executive. Avoid machine learning jargon.
- Do not mention these rules in your answer.
```

**User template**

```text
## Booking under review
{booking_profile}

## Model output (Random Forest, test ROC-AUC 0.78)
- Probability that this booking will be completed: {completion_probability}
- Decision threshold used by the team: {decision_threshold}
- Model decision: {predicted_outcome}
- Completion likelihood tier: {likelihood_tier}
- Average completion rate across all bookings: {baseline_rate}

## Historical context (training data)
{origin_context}
{route_context}

## Strongest model drivers overall, with this booking's values
{top_model_drivers}

## Passenger segment
{segment_profile}

## Task
Write an executive briefing using exactly these five headings, in this order, and nothing else:

**1. Headline**: one sentence (at most 25 words) stating the completion outlook and the main reason.
**2. Outlook**: the probability as a percentage, compared with the average completion rate and with the passenger segment's completion rate, and what the likelihood tier means for the business.
**3. Key factors**: 2 or 3 bullet points of one sentence each. Each bullet covers exactly one detail of this booking, quotes its value from the input, and links it to the historical context, the model drivers or the passenger segment above. Do not link factors to each other or claim relationships between them. At least one bullet must use the passenger segment.
**4. Recommended action**: 1 or 2 concrete commercial actions for bookings like this one, each naming the team that would own it (for example digital sales or ancillary products).
**5. Confidence and caveats**: one or two sentences on how far this estimate can be trusted.

Keep the whole briefing under 180 words.
```

Placeholders: booking_profile, completion_probability, decision_threshold, predicted_outcome, likelihood_tier, baseline_rate, origin_context, route_context, top_model_drivers, segment_profile


### 3.3 Assumptions and limitations

**Assumptions**

- The Random Forest probability is reasonably reliable around the 0.19 threshold chosen in Phase 1.
- Global permutation importance is an acceptable stand-in for per-booking explanations, because Phase 1 did not compute per-booking explanations such as SHAP.
- Historical rates from the training split describe how similar bookings behaved and do not leak test labels.

**Limitations**

- Driver statements describe the model in general, not the exact reasons for this single booking. The model could link a driver to this booking more strongly than the evidence supports. The grounding rules reduce this risk but cannot remove it.
- Origins and routes with few training bookings have unstable rates. The prompt reports the count, but the model may not always mention it.
- Personalization is limited to the booking's own details and its K-Means segment. Segments describe groups, so two different bookings in the same segment receive the same segment text.
- The fixed format is easy to scan but can feel repetitive across many bookings, and it leaves little room for unusual cases.
- Output quality depends on an external API. Answers can change slightly between runs or model versions, and the free tier limits requests per minute.

### 3.4 Example input and output

**Example input.** The filled prompt for Case 2 (borderline booking from Vietnam) is shown below.

In [9]:
def fill(template, case):
    return template["user_template"].format(**{k: case[k] for k in template["placeholders"]})

print(fill(T1, TEST_CASES["Case 2 - Borderline"]))

## Booking under review
- Passengers: 1
- Sales channel: Internet
- Trip type: RoundTrip
- Booked 6 days before departure
- Length of stay: 98 days
- Flight: Wednesday at 07:00 on route HANSYD (8.58 hours)
- Booking origin: Vietnam
- Extra services requested: extra baggage, in-flight meals

## Model output (Random Forest, test ROC-AUC 0.78)
- Probability that this booking will be completed: 25.0%
- Decision threshold used by the team: 19%
- Model decision: Likely to be completed
- Completion likelihood tier: Moderate (above the threshold, but the outcome is uncertain)
- Average completion rate across all bookings: 14.9%

## Historical context (training data)
- Bookings from Vietnam: 28.4% completed (299 training bookings)
- Bookings on route HANSYD: 22.9% completed (35 training bookings)

## Strongest model drivers overall, with this booking's values
1. Booking origin (country the booking was made from): this booking = Vietnam
2. Route: this booking = HANSYD
3. Flight duration: this bo

**Example output.** Gemini's briefing for the same booking (Template 1 v1.2, 193 words, generated in 2.61 seconds). All three outputs are in Section 4 and in `example_outputs/`.

> **1. Headline**
> This booking has a moderate 25.0% completion likelihood, driven primarily by its origin in Vietnam and the requested extra services.
>
> **2. Outlook**
> The 25.0% completion probability exceeds the 14.9% overall average and the passenger segment rate of 23.8%. Sitting in the Moderate tier above our 19% threshold, it is classified as likely to be completed, though the final outcome remains uncertain.
>
> **3. Key factors**
> - The booking origin is "Vietnam", which is the top model driver overall and historically shows a 28.4% completion rate.
> - The passenger belongs to the "Extra-service lovers" segment, a group associated with a 23.8% completion rate that frequently requests extra baggage and meals.
> - The route is "HANSYD", the second strongest model driver, which has a historical completion rate of 22.9%.
>
> **4. Recommended action**
> - The digital sales team should prioritize engagement for Vietnam-origin bookings to protect completion rates.
> - The ancillary products team should ensure seamless delivery of requested extra baggage and meals for this segment.
>
> **5. Confidence and caveats**
> With a test ROC-AUC of 0.78, the model is useful for prioritizing bookings, but it cannot predict the outcome of any single booking with certainty.

## 4. Template 1 Testing

Each test case is filled into T1 and sent to Gemini with the shared settings from Section 1.2. For each case we save:

- `example_outputs/T1_case<n>.md`: the filled prompt and the briefing, for reading.
- `example_outputs/T1_results.json`: the prompt, response, word count and response time for every case. Section 6 reads this file, so T1 and T2 are compared on stored outputs instead of new calls.

If no API key is set, this step is skipped and the earlier saved outputs (if any) are shown.

In [10]:
def run_template(template, cases):
    results = []
    for i, (name, case) in enumerate(cases.items(), 1):
        prompt = fill(template, case)
        text, seconds = generate(template, prompt)
        record = {
            "template_id": template["template_id"], "case": name, "row_id": case["row_id"],
            "model": cfg.MODEL_NAME, "settings": {**cfg.GENERATION_SETTINGS, "thinking_level": cfg.THINKING_LEVEL},
            "prompt": prompt, "response": text, "word_count": len(text.split()),
            "response_seconds": seconds, "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        }
        results.append(record)
        with open(OUTPUTS_DIR / f"{template['template_id']}_case{i}.md", "w", encoding="utf-8") as f:
            f.write(f"# {template['template_id']} | {name} (row {case['row_id']})\n\n"
                    f"Model: `{cfg.MODEL_NAME}` | Generated: {record['generated_at']}\n\n"
                    f"## Prompt\n\n```text\n{prompt}\n```\n\n## Response\n\n{text}\n")
        time.sleep(5)  # stay well inside the free-tier request rate
    with open(OUTPUTS_DIR / f"{template['template_id']}_results.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    return results

results_file = OUTPUTS_DIR / "T1_results.json"
if API_READY:
    T1_RESULTS = run_template(T1, TEST_CASES)
elif results_file.exists():
    T1_RESULTS = json.loads(results_file.read_text(encoding="utf-8"))
    print("No API key: showing the saved outputs.")
else:
    T1_RESULTS = []
    print("No API key and no saved outputs yet. Add GEMINI_API_KEY (Section 1.3) and rerun this cell.")

for r in T1_RESULTS:
    display(Markdown(f"### {r['case']} (row {r['row_id']}, {r['word_count']} words, {r['response_seconds']} s)\n\n{r['response']}\n\n---"))

### Case 1 - Low (row 9669, 185 words, 2.37 s)

**1. Headline**
This booking has a low 5.0% completion likelihood, primarily driven by its origin in Australia and the DMKOOL route.

**2. Outlook**
The booking's 5.0% completion probability falls well below the 14.9% overall average and the "Top market (Australia)" segment's 5.4% completion rate. Being in the low likelihood tier means this booking is expected to be abandoned.

**3. Key factors**
- The booking origin is Australia, which historical training data shows has a low 5.1% completion rate across 14,274 bookings.
- The flight operates on route DMKOOL, which historically sees only a 6.1% completion rate across 538 training bookings.
- The passenger requested 1 extra service (extra baggage), aligning with the "Top market (Australia)" segment where 73% of bookings request extra baggage.

**4. Recommended action**
- The digital sales team should design targeted engagement strategies for long-lead bookings originating in Australia.
- The ancillary products team should optimize service offerings for passengers booking extra baggage on long-duration routes.

**5. Confidence and caveats**
With a test ROC-AUC of 0.78, the model is useful for prioritizing bookings, but it is not certain for any single booking.

---

### Case 2 - Borderline (row 11278, 193 words, 2.61 s)

**1. Headline**
This booking has a moderate 25.0% completion likelihood, driven primarily by its origin in Vietnam and the requested extra services.

**2. Outlook**
The 25.0% completion probability exceeds the 14.9% overall average and the passenger segment rate of 23.8%. Sitting in the Moderate tier above our 19% threshold, it is classified as likely to be completed, though the final outcome remains uncertain.

**3. Key factors**
- The booking origin is "Vietnam", which is the top model driver overall and historically shows a 28.4% completion rate.
- The passenger belongs to the "Extra-service lovers" segment, a group associated with a 23.8% completion rate that frequently requests extra baggage and meals.
- The route is "HANSYD", the second strongest model driver, which has a historical completion rate of 22.9%.

**4. Recommended action**
- The digital sales team should prioritize engagement for Vietnam-origin bookings to protect completion rates.
- The ancillary products team should ensure seamless delivery of requested extra baggage and meals for this segment.

**5. Confidence and caveats**
With a test ROC-AUC of 0.78, the model is useful for prioritizing bookings, but it cannot predict the outcome of any single booking with certainty.

---

### Case 3 - High (row 35092, 184 words, 2.14 s)

**1. Headline**
This booking has a 55.0% completion probability, driven primarily by historical patterns associated with its Malaysian booking origin and AKLKUL route.

**2. Outlook**
The 55.0% completion probability is well above the 14.9% overall average and the 21.1% segment rate, placing it in the High likelihood tier.

**3. Key factors**
- The booking origin is Malaysia, which historically sees a 34.4% completion rate and is the model's top overall driver.
- The route is AKLKUL, which is the model's second strongest driver and historically has a 21.1% completion rate.
- This booking requests 3 extra services, whereas the "Top route (AKLKUL)" segment typically requests preferred seats 23% of the time and meals 45% of the time.

**4. Recommended action**
- The ancillary products team should ensure all 3 requested extra services are seamlessly confirmed to support this high-value booking.
- The digital sales team should monitor long lead-time bookings to maintain customer engagement prior to departure.

**5. Confidence and caveats**
With a test ROC-AUC of 0.78, this model is useful for prioritising bookings, but its estimate is not certain for any single booking.

---

### 4.1 Iteration from version 1.0 to 1.2

T1 was improved twice, each time on the same three cases. The outputs of each earlier version are kept as evidence in `example_outputs/T1_v1.0/` and `example_outputs/T1_v1.1/`.

**v1.0 to v1.1.** Reading the v1.0 outputs showed four problems:

| Problem in v1.0 | Cases | Likely cause | Change in v1.1 |
| --- | --- | --- | --- |
| The briefing said the passenger segment was **missing**, although the prompt contained it | 2 and 3 | The system rule said "if information is missing (for example the passenger segment)...". The example primed the model to expect the segment to be missing. | Removed the example and added "never describe information as missing when it is given" |
| The segment was mostly ignored | 2 and 3 | Nothing in the task asked the model to use it | Outlook must compare with the segment's completion rate, and at least one key factor must use the segment |
| Over 200 words (204 and 248 words) | 1 and 2 | The model counts words loosely | One sentence per key factor, and a limit of 180 words |
| "0.78 ROC-AUC ... reliable performance" overstated the model | 1 | No guidance on describing accuracy | A rule to describe accuracy honestly: useful for prioritising, not certain for one booking |

The v1.0 facts were otherwise correct: every number and booking detail in the briefings could be traced to the prompt. **v1.1 to v1.2.** All v1.1 briefings passed the automatic checks, but reading them showed one confusing sentence in the Malaysia case: *"Requesting 3 extra services aligns with key drivers like length of stay and flight duration."* The model had joined unrelated drivers on its own. v1.2 requires each key factor to cover exactly one booking detail, quote its value, and not link factors to each other. In the v1.2 run, every key factor names one detail with its value, for example: *"The route is AKLKUL, which is the model's second strongest driver and historically has a 21.1% completion rate."*

This second fix was found by reading, not by the automatic checks, which shows that the checks cannot replace a human review.

The changes and their reasons are also recorded in the template file (`changes_from_1.0` and `changes_from_1.1`).

### 4.2 Format and grounding check

A quick automatic check that each T1 briefing does what the template asks. Full quality scoring (relevance, readability, keyword alignment and preference simulation) is part of the comparison in Section 6.

- **All five headings** appear, in order.
- **Under 200 words.**
- **The probability is quoted correctly**: the percentage in the briefing matches the model output, rounded to whole percent.
- **No price advice**: the words "price", "fare" and "discount" do not appear.
- **Segment used**: the segment's name appears, and the briefing does not call the segment missing.

In [11]:
HEADINGS = ["Headline", "Outlook", "Key factors", "Recommended action", "Confidence and caveats"]

def check(record, case):
    text = record["response"]
    positions = [text.lower().find(h.lower()) for h in HEADINGS]
    prob = float(case["completion_probability"].rstrip("%"))
    quoted = {round(float(x)) for x in re.findall(r"(\d+(?:\.\d+)?)\s*%", text)}
    return {
        "case": record["case"],
        "all headings in order": all(p >= 0 for p in positions) and positions == sorted(positions),
        "under 200 words": record["word_count"] < 200,
        "probability quoted": round(prob) in quoted,
        "no price advice": not re.search(r"\b(price|pricing|fare|discount)", text, re.I),
        "segment used": (segment_name(case) or "").lower() in text.lower()
                        and not re.search(r"segment[^.]*(missing|not (provided|available))", text, re.I),
    }

def segment_name(case):
    match = re.match(r'"([^"]+)"', case["segment_profile"])
    return match.group(1) if match else None

if T1_RESULTS:
    display(pd.DataFrame([check(r, TEST_CASES[r["case"]]) for r in T1_RESULTS]).set_index("case"))

,all headings in order,under 200 words,probability quoted,no price advice,segment used
case,,,,,
Case 1 - Low,True,True,True,True,True
Case 2 - Borderline,True,True,True,True,True
Case 3 - High,True,True,True,True,True


### 4.3 Observations on Template 1

- **Final result (v1.2):** all three briefings followed the five headings, stayed under 200 words, used only numbers from the prompt, and passed every automatic check.
- **Personalization:** the advice changed with each booking, and the segment made it specific to the customer (for example, confirming baggage and meals for the "Extra-service lovers" booking).
- **Iterative testing improved quality:** testing on the same three cases found two issues that we fixed: a false "segment missing" statement (fixed in v1.1) and a sentence that joined unrelated factors (fixed in v1.2). The second issue was found by reading the outputs, so the final design combines automatic checks with human review.
- **Lesson learned:** prompt wording has a strong effect on the output, so every change to the template was retested on the same cases before it was accepted.

## 5. Template 2 Design Documentation

*To be completed by Member D.*

Use the same documentation fields as Section 3 (name and ID, use case, rationale, structure with placeholders, example input/output, assumptions and limitations). Save the template as `prompts/T2_<name>.json` with the same keys as T1 (`system_instruction`, `user_template`, `placeholders`), so `fill()` and `run_template()` work unchanged. Reuse the three cases in `TEST_CASES` (also saved in `example_outputs/test_cases.json`).

## 6. Testing Framework & Output Comparison

*To be completed by Member D.*

Run `run_template(T2, TEST_CASES)` and compare against `example_outputs/T1_results.json`.

## 7. Analysis: Qualitative & Quantitative Results

*To be completed by Member D.*

Qualitative: relevance, completeness, clarity, personalization, safety and hallucinations. Quantitative: response length, keyword alignment, Flesch-Kincaid readability, simulated user preference.

## 8. Best Prompt Selection & Justification

*To be completed by Member D.*

## 9. Integration Plan for the Final System

*To be completed by Members D and E.*

## 10. Ethical Considerations & Limitations

*To be completed by Member D.*